# Kiểm thử CNN-only (`max_len=1024`) trên hai dataset mới

Notebook này **không train lại** mô hình khi chạy test. Nó gộp `obfuscated_grouped.csv` và `xss_payloads_with_obfuscated.csv`, tiền xử lý bằng đúng pipeline dùng ở notebook CNN–LSTM, rồi đánh giá bằng **CNN-only đã train lại với `max_len=1024`**. Ô cuối in bảng hiệu quả với cùng các cột để so sánh.

- Nhãn: `0 = Normal`, `1 = Attack`.
- Chỉ bỏ bản ghi gốc trùng **hoàn toàn trong từng dataset** và payload rỗng; giữ các biến thể obfuscation.
- Payload được bọc bằng `wrap_payload_as_request`, sau đó chuẩn hóa khoảng trắng như lúc train. Không URL decode, HTML unescape hay lowercase.
- Model CNN-only mới dùng `max_len=1024`, tokenizer fit trên cùng train split `obfu_http` và threshold `0.5`, giống điều kiện đầu vào của model webapp.
- Test trên toàn bộ dữ liệu theo mặc định; không fit hay tune trên dữ liệu mới.

## 1. Thiết lập

Chạy notebook từ thư mục gốc project hoặc thư mục `cnn_only`. Notebook dùng Python trong `.venv-webapp` nếu có; nếu không, dùng Python của kernel. Môi trường chạy model cần các thư viện trong `webapp/requirements.txt`.

Checkpoint CNN-only được train riêng bằng lệnh sau, không ghi đè bản `max_len=768` cũ:

```bash
python cnn_only/train_cnn_only.py --datasets obfu_http --train-sources obfu_http --obfuscation-path obfu_http_dataset_v2.csv --output-dir cnn_only/artifacts_cnn_only_matched_1024 --max-len 1024 --epochs 50 --batch-size 128 --seed 42 --split-protocol random_stratified_row
```

Train và validation CSV của lượt train này trùng byte với các split `obfu_http` đã lưu cho CNN–LSTM.

In [1]:
import json
import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "preprocessing" / "preprocess_data.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Hãy chạy notebook từ repo root hoặc thư mục cnn_lstm.")

MODEL_PYTHON_CANDIDATES = [
    PROJECT_ROOT / ".venv-webapp" / "Scripts" / "python.exe",  # Windows
    PROJECT_ROOT / ".venv-webapp" / "bin" / "python",          # Linux/macOS
]
MODEL_PYTHON = next((path for path in MODEL_PYTHON_CANDIDATES if path.is_file()), Path(sys.executable))
EVALUATOR = PROJECT_ROOT / "analysis" / "evaluate_merged_cnn_only.py"
OUTPUT_DIR = PROJECT_ROOT / "reports" / "merged_external_evaluation" / "cnn_only"
INPUT_FILES = [
    PROJECT_ROOT / "obfuscated_grouped.csv",
    PROJECT_ROOT / "xss_payloads_with_obfuscated.csv",
]
MODEL_FILE = PROJECT_ROOT / "cnn_only" / "artifacts_cnn_only_matched_1024" / "by_dataset" / "obfu_http" / "best_cnn_only.keras"

missing = [path for path in [EVALUATOR, MODEL_FILE, *INPUT_FILES] if not path.is_file()]
if missing:
    raise FileNotFoundError("Thiếu file cần thiết:\n" + "\n".join(map(str, missing)))

BATCH_SIZE = 512
TEST_LIMIT = None  # Đặt số nguyên dương để chạy thử một phần; None = toàn bộ dữ liệu.
print("Project:", PROJECT_ROOT)
print("Python chạy model:", MODEL_PYTHON)
print("Model:", MODEL_FILE.name)
print("Giới hạn test:", TEST_LIMIT or "Toàn bộ dữ liệu")

Project: C:\Users\admin\Desktop\obfuscated-web-attack-detection
Python chạy model: C:\Users\admin\Desktop\obfuscated-web-attack-detection\.venv-webapp\Scripts\python.exe
Model: best_cnn_only.keras
Giới hạn test: Toàn bộ dữ liệu


## 2. Gộp, tiền xử lý và chạy test

Ô này gọi `analysis/evaluate_merged_cnn_only.py` với checkpoint CNN-only vừa train ở `max_len=1024`. Phần gộp và tiền xử lý dùng chung hàm với notebook CNN–LSTM. Kết quả nằm trong `reports/merged_external_evaluation/cnn_only/`; threshold lấy từ metadata của CNN-only (hiện là `0.5`).

In [2]:
command = [
    str(MODEL_PYTHON), str(EVALUATOR),
    "--batch-size", str(BATCH_SIZE),
    "--output-dir", str(OUTPUT_DIR),
]
if TEST_LIMIT is not None:
    if not isinstance(TEST_LIMIT, int) or TEST_LIMIT <= 0:
        raise ValueError("TEST_LIMIT phải là số nguyên dương hoặc None.")
    command += ["--limit", str(TEST_LIMIT)]

with subprocess.Popen(
    command,
    cwd=PROJECT_ROOT,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    encoding="utf-8",
    errors="replace",
    bufsize=1,
) as process:
    for line in process.stdout:
        print(line, end="")
    return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f"Đánh giá thất bại (exit code {return_code}).")

2026-09-17 22:26:39.737596: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


2026-09-17 22:26:40.802119: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


2026-09-17 22:26:59.878015: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Evaluated 10,240/156,186 rows


Evaluated 20,480/156,186 rows


...

Evaluated 133,120/156,186 rows


Evaluated 143,360/156,186 rows


Evaluated 153,600/156,186 rows


Evaluated 156,186/156,186 rows


Saved merged data: C:\Users\admin\Desktop\obfuscated-web-attack-detection\reports\merged_external_evaluation\cnn_only\merged_preprocessed.csv
Saved predictions: C:\Users\admin\Desktop\obfuscated-web-attack-detection\reports\merged_external_evaluation\cnn_only\predictions.csv
Saved metrics: C:\Users\admin\Desktop\obfuscated-web-attack-detection\reports\merged_external_evaluation\cnn_only\summary.json
Overall: {"rows": 156186, "label_counts": {"normal": 74106, "attack": 82080}, "accuracy": 0.8078572983494039, "confusion_matrix_tn_fp_fn_tp": [57269, 16837, 13173, 68907], "normal": {"precision": 0.8129950881576332, "recall": 0.772798423879308, "f1-score": 0.7923873038713783, "support": 74106.0}, "attack": {"precision": 0.8036364060459041, "recall": 0.8395102339181286, "f1-score": 0.8211817141767566, "support": 82080.0}, "macro_avg": {"precision": 0.8083157471017687, "recall": 0.8061543288987183, "f1-score": 0.8067845090240675, "support": 156186.0}, "weighted_avg": {"precision": 0.808076845

## 3. Kiểm tra dữ liệu sau tiền xử lý

Kiểm tra số dòng được giữ lại, số dòng trùng hoàn toàn bị bỏ trong từng file và mức trùng chính xác với tập train đã lưu. Các dòng có cùng đầu vào sau chuẩn hóa vẫn được giữ theo yêu cầu.

In [3]:
with (OUTPUT_DIR / "summary.json").open(encoding="utf-8") as file:
    results = json.load(file)

prep = results["preparation"]
source_audit = pd.DataFrame([
    {
        "Dataset": source,
        "Dòng gốc": count,
        "Trùng hoàn toàn đã bỏ": prep["exact_duplicate_rows_removed_by_source"][source],
        "Dòng đã test": prep["evaluated_rows_by_source"][source],
    }
    for source, count in prep["input_rows_by_source"].items()
])
display(source_audit)
print("Payload rỗng đã bỏ:", prep["empty_rows_removed"])
print("Đầu vào giống nhau sau chuẩn hóa vẫn giữ:", prep["repeated_model_inputs_retained"])
print("Trùng chính xác với tập train:", results["training_overlap_rows"] if results["training_overlap_checked"] else "Không có tập train để kiểm tra")
print("Bị cắt ở max_len:", results["truncated_rows"])
print("Đây là kết quả một phần:" , results["partial_test"])

,Dataset,Dòng gốc,Trùng hoàn toàn đã bỏ,Dòng đã test
0,obfuscated_grouped,134778,0,134777
1,xss_payloads_with_obfuscated,21410,0,21409


Payload rỗng đã bỏ: 2
Đầu vào giống nhau sau chuẩn hóa vẫn giữ: 10687
Trùng chính xác với tập train: 0
Bị cắt ở max_len: 5284
Đây là kết quả một phần: False


## 4. Hiệu quả theo kỹ thuật biến đổi

Recall của lớp Attack theo từng kỹ thuật giúp thấy kiểu obfuscation nào mô hình dễ bỏ sót. `unspecified` là dataset XSS không có cột `technique`.

In [4]:
technique_rows = []
for technique, result in results["by_technique"].items():
    matrix = result["confusion_matrix_tn_fp_fn_tp"]
    technique_rows.append({
        "Technique": technique,
        "Rows": result["rows"],
        "Attack Recall": result["attack"]["recall"],
        "Attack F1": result["attack"]["f1-score"],
        "FP": matrix[1],
        "FN": matrix[2],
    })
technique_table = pd.DataFrame(technique_rows).sort_values("Technique").reset_index(drop=True)
display(technique_table.style.format({"Attack Recall": "{:.4%}", "Attack F1": "{:.4%}"}))

,Technique,Rows,Attack Recall,Attack F1,FP,FN
0,case_swapping,7335,77.7641%,87.4914%,0,1631
1,comment_injection,9931,82.5395%,90.4347%,0,1734
2,comment_rewriting,1077,71.6806%,83.5046%,0,305
3,integer_encoding,9462,84.6227%,91.6710%,0,1455
4,logical_invariant,5059,82.0913%,90.1650%,0,906
5,none,80575,78.2300%,46.6585%,15710,2177
6,operator_swapping,6333,73.8355%,84.9487%,0,1657
7,tautology_change,5059,87.8237%,93.5172%,0,616
8,unspecified,21409,96.9068%,95.3757%,1127,553
9,whitespace_substitution,9946,78.4939%,87.9513%,0,2139


## 5. Bảng hiệu quả cuối cùng

Bảng dùng cùng cột với notebook CNN–LSTM: Accuracy, AUC, PR-AUC, threshold, Precision/Recall/F1 của Attack, FP và FN. Dòng `merged_all` là kết quả chính; hai dòng còn lại cho biết hiệu quả theo nguồn.

In [5]:
def performance_row(dataset_name, result):
    tn, fp, fn, tp = result["confusion_matrix_tn_fp_fn_tp"]
    return {
        "Model": "CNN-only (max_len=1024)",
        "Dataset": dataset_name,
        "Rows": result["rows"],
        "Accuracy": result["accuracy"],
        "AUC-ROC": result.get("roc_auc"),
        "PR-AUC": result.get("average_precision"),
        "Threshold": results["threshold"],
        "Attack Precision": result["attack"]["precision"],
        "Attack Recall": result["attack"]["recall"],
        "Attack F1": result["attack"]["f1-score"],
        "FP": fp,
        "FN": fn,
    }

performance_rows = [performance_row("merged_all", results["overall"])]
performance_rows.extend(
    performance_row(name, result) for name, result in results["by_source"].items()
)
performance_table = pd.DataFrame(performance_rows)
performance_table.to_csv(OUTPUT_DIR / "performance_table.csv", index=False, encoding="utf-8")

display(Markdown("### Hiệu quả CNN-only trên dataset đã gộp"))
display(performance_table.style.format({
    "Accuracy": "{:.4%}",
    "AUC-ROC": "{:.4%}",
    "PR-AUC": "{:.4%}",
    "Threshold": "{:.4f}",
    "Attack Precision": "{:.4%}",
    "Attack Recall": "{:.4%}",
    "Attack F1": "{:.4%}",
}))

### Hiệu quả CNN-only trên dataset đã gộp

,Model,Dataset,Rows,Accuracy,AUC-ROC,PR-AUC,Threshold,Attack Precision,Attack Recall,Attack F1,FP,FN
0,CNN-only (max_len=1024),merged_all,156186,80.7857%,89.1840%,89.0011%,0.5000,80.3636%,83.9510%,82.1182%,16837,13173
1,CNN-only (max_len=1024),obfuscated_grouped,134777,78.9801%,87.3547%,84.3754%,0.5000,76.6540%,80.3433%,78.4553%,15710,12620
2,CNN-only (max_len=1024),xss_payloads_with_obfuscated,21409,92.1528%,96.5453%,99.2520%,0.5000,93.8923%,96.9068%,95.3757%,1127,553


## 6. So sánh trực tiếp với CNN–LSTM webapp

Ô cuối chỉ tạo bảng nếu hai lượt test có **cùng số dòng, threshold, `max_len=1024` và file đầu vào sau tiền xử lý giống hệt nhau**. Hãy chạy notebook CNN–LSTM trước để có `summary.json` của model webapp.

In [6]:
comparison_script = PROJECT_ROOT / "analysis" / "compare_merged_models.py"
subprocess.run([str(MODEL_PYTHON), str(comparison_script)], cwd=PROJECT_ROOT, check=True)
comparison_table = pd.read_csv(OUTPUT_DIR.parent / "comparison_matched_1024.csv")

display(Markdown("### So sánh hai model trên cùng dataset và max_len=1024"))
display(comparison_table.style.format({
    "Threshold": "{:.4f}",
    "Accuracy": "{:.4%}",
    "AUC-ROC": "{:.4%}",
    "PR-AUC": "{:.4%}",
    "Attack Precision": "{:.4%}",
    "Attack Recall": "{:.4%}",
    "Attack F1": "{:.4%}",
    "Truncated": "{:.0f}",
}, na_rep=""))

### So sánh hai model trên cùng dataset và max_len=1024

,Model,Dataset,Rows,Max_len,Threshold,Accuracy,AUC-ROC,PR-AUC,Attack Precision,Attack Recall,Attack F1,FP,FN,Truncated
0,CNN-LSTM (WebApp),merged_all,156186,1024,0.5000,80.9509%,90.3319%,89.7560%,87.3110%,74.5931%,80.4526%,8898,20854,5284
1,CNN-LSTM (WebApp),obfuscated_grouped,134777,1024,0.5000,79.0098%,88.8527%,84.4628%,84.3353%,68.6957%,75.7163%,8192,20098,
2,CNN-LSTM (WebApp),xss_payloads_with_obfuscated,21409,1024,0.5000,93.1711%,96.5778%,99.3744%,96.0399%,95.7713%,95.9055%,706,756,
3,CNN-only (1024),merged_all,156186,1024,0.5000,80.7857%,89.1840%,89.0011%,80.3636%,83.9510%,82.1182%,16837,13173,5284
4,CNN-only (1024),obfuscated_grouped,134777,1024,0.5000,78.9801%,87.3547%,84.3754%,76.6540%,80.3433%,78.4553%,15710,12620,
5,CNN-only (1024),xss_payloads_with_obfuscated,21409,1024,0.5000,92.1528%,96.5453%,99.2520%,93.8923%,96.9068%,95.3757%,1127,553,
